# NOAA IBTrACS: inspect a tropical cyclone track

IBTrACS v04r01 combines best tracks from different agencies. This example selects Hurricane Milton (SID `2024279N21265`) from NOAA's rolling `last3years` subset and preserves the original observation fields. The subset changes over time; its version does not pin its bytes. Record the SHA-256 and retain the downloaded snapshot for reproducibility. Basin subsets can overlap, and recent tracks may be provisional. Track points and reported storm winds are not asset winds, return-period curves, or impacts. Wind averaging periods can differ by agency; consult the [NOAA documentation](https://www.ncei.noaa.gov/products/international-best-track-archive) before comparing values.

The companion pipeline is `pipelines/ibtracs_pipeline.py`. It downloads the full selected CSV, then streams filtered rows into Parquet; filtering does not reduce the download size.

Run from a local crc-docs checkout: use `uv sync`, then launch Jupyter with
`uv run jupyter lab` and select the project kernel. The first pipeline run
requires internet access.


In [1]:
# Check the local notebook environment before running the pipeline.
import importlib.util
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (root / "pipelines").is_dir():
    raise RuntimeError("Run this notebook from the crc-docs checkout or its notebooks directory.")
missing = [name for name in ("crc_sdk", "pyarrow", "pandas")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Run `uv sync` from the crc-docs checkout and select its kernel: {missing}")
sys.path.insert(0, str(root))
from pipelines.ibtracs_pipeline import run
import pyarrow.parquet as pq

output = Path("/tmp/crc-docs-ibtracs")
summary = run(output, "2024279N21265")
print("Track points:", summary["track_points"])
print("Snapshot SHA-256:", summary["source"]["sha256"])
print("Retrieved:", summary["source"]["retrieved_at"])

Track points: 62
Snapshot SHA-256: f4b8eb58f5f4ac853f907d123d93215ea709e99c57142454cc67f9114c30143a
Retrieved: 2026-10-30T17:43:19.580670+00:00


In [2]:
track = pq.read_table(output / "track.parquet", columns=["SID", "ISO_TIME", "LAT", "LON", "USA_WIND", "USA_PRES"])
print("Units:", {key: summary["source"]["units"][key] for key in ["LAT", "LON", "USA_WIND", "USA_PRES"]})
track.slice(0, 5).to_pandas()

Units: {'LAT': 'degrees_north', 'LON': 'degrees_east', 'USA_WIND': 'kts', 'USA_PRES': 'mb'}


,SID,ISO_TIME,LAT,LON,USA_WIND,USA_PRES
0,2024279N21265,2024-10-04 18:00:00,21.0,-94.6,30,1009
1,2024279N21265,2024-10-04 21:00:00,20.8,-94.9,30,1009
2,2024279N21265,2024-10-05 00:00:00,20.8,-95.2,30,1008
3,2024279N21265,2024-10-05 03:00:00,21.0,-95.3,30,1008
4,2024279N21265,2024-10-05 06:00:00,21.4,-95.4,30,1008


Values are NOAA's strings and blank fields are null. `USA_WIND` is the US agency's reported maximum sustained wind in knots, not a wind speed at a particular asset. The snapshot manifest records the exact source URL, retrieval time, HTTP validators and checksum. When the rolling subset no longer includes this storm, select a suitable basin or the full archive instead.